Save base Logit, KNN, SVC, LogisticRegression, CatBoost, XGboost with pipeline. Sedimentation_Stk feature was used!

Based on `research_14_SMOTE_influence_on_metrics.ipynb` it was decided to use SMOTE for all models, except Logit!

base_estimator_params - redundant thing in logging!

# Import libraries

In [1]:
# Changes to all modules will automatically be applied when any cell runs. 
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

import joblib

from statsmodels.api import Logit
from statsmodels.api import Logit # type: ignore
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier

from sklearn.metrics import (
    make_scorer,
    f1_score,
)

import optuna
from functools import partial

from pathlib import Path
import sys

from typing import Type
from sklearn.base import BaseEstimator

sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from split_tools import get_train_test
from modelling4_utils import (
    # _prepare_df,
    MLPipeline,
    StatsModelsEstimator,
    update_estimator_params,
    OptunaOptimizer,
    GridSearchOptimizer,
    RANDOM_STATE,
)

seed = RANDOM_STATE

In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Settings

In [4]:
model_postfix = 'opt-model'
add_smote = False
is_smotenc = False
smote_params = {
    # 'categorical_features': (
    #     'wettability',
    #     'inclination',
    # ),
}
scoring_metrics = {"f1_macro": make_scorer(f1_score, average="macro"),}
step_scoring_average = "mean"
n_trials = 40 # was checked with 1000 and 5000

save_model_and_metrics = False

## Optimization functions

In [5]:
def optimize_estimator_optuna(
    target:str,
    estimator:Type[BaseEstimator],
    estimator_params:dict,
    objective:callable,
    n_trials:int,
    step_scoring_average:str=step_scoring_average,
    params_to_process:list=None,
    model_postfix:str=model_postfix,
    add_smote:bool=add_smote,
    is_smotenc:bool=is_smotenc,
    smote_params:dict=smote_params,
    scoring_metrics:dict=scoring_metrics,
    save_model_and_metrics:bool=save_model_and_metrics,
    seed:int=seed,
):
    
    # Switch off probability for SVC, since it is long to optimize
    if 'probability' in estimator_params:
        estimator_params['probability'] = False
    
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=smote_params,
        scoring_metrics=scoring_metrics,
        step_scoring_average=step_scoring_average,
    )
    
    opt = OptunaOptimizer(
        objective=partial(objective, ml_pipe=ml_pipe),
        study_name="model_study",
        direction="maximize",
        seed=seed,
    )
    
    opt.optimize(n_trials=n_trials)
    
    best_params = opt.study.best_params
    
    if params_to_process:
        for param in params_to_process:
            # Find one key in best_params which contains param
            key = next((k for k in best_params.keys() if param in k), None)
            if key:
                best_params[param] = best_params.pop(key)

    print('best_params')
    display(best_params)
    
    # Switch on probability for SVC, to get metrics like roc_auc for the final model
    if 'probability' in estimator_params:
        estimator_params['probability'] = True
    
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params={**estimator_params, **best_params},
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=smote_params,
        step_scoring_average=step_scoring_average, # No need to pass, it would not be used
    )
    
    ml_pipe.run(
        save_model_and_metrics=save_model_and_metrics,
    )

# CatBoost

In [6]:
estimator = CatBoostClassifier
target = 'splashing'
# TODO: Try with GPU!
estimator_params = {
    'verbose': False,
}
# params_to_process = ['gamma']
params_to_process = None

def catboost_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):

    suggested_estimator_params = {
        "iterations": trial.suggest_int("iterations", 50, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "depth": trial.suggest_int("depth", 3, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.0, 1.0),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "grow_policy": trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]),
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]),
        "auto_class_weights": trial.suggest_categorical("auto_class_weights", ["Balanced", "SqrtBalanced", None]),
    }
    
    if suggested_estimator_params['bootstrap_type'] == 'Bernoulli':
        suggested_estimator_params['subsample'] = trial.suggest_float(
            "subsample", 0.5, 1.0
        )
        
    if suggested_estimator_params['bootstrap_type'] == 'Bayesian':
        suggested_estimator_params['bagging_temperature'] = (
            trial.suggest_float("bagging_temperature", 0.0, 1.0)
        )

    estimator_params = update_estimator_params(
        ml_pipe,
        suggested_estimator_params,
    )
    
    score = ml_pipe.step(
        estimator=ml_pipe._pipeline_params['estimator'],
        estimator_params=estimator_params,
    )
    
    return score

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=catboost_objective,
    n_trials=n_trials,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-15 12:08:19,211] A new study created in memory with name: model_study
[I 2025-04-15 12:08:25,749] Trial 0 finished with value: 0.8937955000766388 and parameters: {'iterations': 406, 'learning_rate': 0.22648248189516848, 'depth': 8, 'l2_leaf_reg': 0.24810409748678125, 'random_strength': 0.15601864044243652, 'border_count': 66, 'grow_policy': 'Depthwise', 'bootstrap_type': 'MVS', 'auto_class_weights': 'Balanced'}. Best is trial 0 with value: 0.8937955000766388.
[I 2025-04-15 12:08:28,277] Trial 1 finished with value: 0.8768204014310234 and parameters: {'iterations': 224, 'learning_rate': 0.005670807781371429, 'depth': 7, 'l2_leaf_reg': 0.05342937261279776, 'random_strength': 0.2912291401980419, 'border_count': 169, 'grow_policy': 'Lossguide', 'bootstrap_type': 'Bernoulli', 'auto_class_weights': 'SqrtBalanced', 'subsample': 0.8037724259507192}. Best is trial 0 with value: 0.8937955000766388.
[I 2025-04-15 12:08:35,351] Trial 2 finished with value: 0.8641856475089463 and paramet

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "CatBoostClassifier"


,0
target,splashing
model,CatBoostClassifier_splashing_opt-model
holdout_test_f1_macro,0.823391
holdout_test_accuracy_balanced,0.818287
holdout_test_roc_auc,0.94213
holdout_test_f1,0.877551
holdout_test_accuracy,0.84
cv_test_f1_macro_median,0.896201
cv_test_accuracy_balanced_median,0.891641
cv_test_roc_auc_median,0.947368


In [7]:
estimator = CatBoostClassifier
target = 'no_fragmentation'
estimator_params = {
    'verbose': False,
}
# params_to_process = ['gamma']
params_to_process = None

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=catboost_objective,
    n_trials=n_trials,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-15 10:50:03,125] A new study created in memory with name: model_study
[I 2025-04-15 10:50:16,470] Trial 0 finished with value: 0.855167909848696 and parameters: {'iterations': 500, 'learning_rate': 0.22648248189516848, 'depth': 9, 'l2_leaf_reg': 6.387926357773329, 'random_strength': 0.15601864044243652, 'border_count': 66, 'grow_policy': 'Depthwise', 'bootstrap_type': 'MVS'}. Best is trial 0 with value: 0.855167909848696.
[I 2025-04-15 10:50:22,718] Trial 1 finished with value: 0.8436344979961399 and parameters: {'iterations': 866, 'learning_rate': 0.0033572967053517922, 'depth': 5, 'l2_leaf_reg': 2.650640588680904, 'random_strength': 0.3042422429595377, 'border_count': 149, 'grow_policy': 'Lossguide', 'bootstrap_type': 'MVS'}. Best is trial 0 with value: 0.855167909848696.
[I 2025-04-15 10:50:24,165] Trial 2 finished with value: 0.8515161679441593 and parameters: {'iterations': 565, 'learning_rate': 0.08810003129071789, 'depth': 5, 'l2_leaf_reg': 5.628109945722504, 'random_

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "CatBoostClassifier"


,0
target,no_fragmentation
model,CatBoostClassifier_no_fragmentation_opt-model
holdout_test_f1_macro,0.949653
holdout_test_accuracy_balanced,0.956818
holdout_test_roc_auc,0.99
holdout_test_f1,0.926829
holdout_test_accuracy,0.96
cv_test_f1_macro_median,0.903571
cv_test_accuracy_balanced_median,0.887179
cv_test_roc_auc_median,0.982906


# XGBoost

In [8]:
estimator = XGBClassifier
target = 'splashing'
estimator_params = {}
# params_to_process = ['gamma']
params_to_process = None

def xgboost_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):

    # https://xgboost.readthedocs.io/en/latest/parameter.html
    # sum(negative instances) / sum(positive instances)
    scale_pos_weight = 132/240 # For splashing

    suggested_estimator_params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True), # Catboost l2_leaf_reg
        "scale_pos_weight": trial.suggest_categorical("scale_pos_weight", [scale_pos_weight, 1.0]),
    }

    estimator_params = update_estimator_params(
        ml_pipe,
        suggested_estimator_params,
    )
    
    score = ml_pipe.step(
        estimator=ml_pipe._pipeline_params['estimator'],
        estimator_params=estimator_params,
    )
    
    return score

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=xgboost_objective,
    n_trials=n_trials,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-15 12:12:21,693] A new study created in memory with name: model_study
[I 2025-04-15 12:12:22,030] Trial 0 finished with value: 0.7629192547654765 and parameters: {'n_estimators': 406, 'learning_rate': 0.22648248189516848, 'max_depth': 8, 'min_child_weight': 12, 'gamma': 0.7800932022121826, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.5290418060840998, 'reg_alpha': 2.9154431891537547, 'reg_lambda': 0.2537815508265665}. Best is trial 0 with value: 0.7629192547654765.
[I 2025-04-15 12:12:22,833] Trial 1 finished with value: 0.392616568979092 and parameters: {'n_estimators': 723, 'learning_rate': 0.001124579825911934, 'max_depth': 10, 'min_child_weight': 17, 'gamma': 1.0616955533913808, 'subsample': 0.5909124836035503, 'colsample_bytree': 0.5917022549267169, 'reg_alpha': 0.016480446427978974, 'reg_lambda': 0.12561043700013558}. Best is trial 0 with value: 0.7629192547654765.
[I 2025-04-15 12:12:23,774] Trial 2 finished with value: 0.8577090282829423 and parameters: {'n

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "XGBClassifier"


,0
target,splashing
model,XGBClassifier_splashing_opt-model
holdout_test_f1_macro,0.868703
holdout_test_accuracy_balanced,0.865741
holdout_test_roc_auc,0.945216
holdout_test_f1,0.907216
holdout_test_accuracy,0.88
cv_test_f1_macro_median,0.860788
cv_test_accuracy_balanced_median,0.873839
cv_test_roc_auc_median,0.942724


In [9]:
estimator = XGBClassifier
target = 'no_fragmentation'
estimator_params = {}
# params_to_process = ['gamma']
params_to_process = None

def xgboost_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):

    # https://xgboost.readthedocs.io/en/latest/parameter.html
    # sum(negative instances) / sum(positive instances)
    scale_pos_weight = 273/99 # For no_fragmentation

    suggested_estimator_params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True), # Catboost l2_leaf_reg
        "scale_pos_weight": trial.suggest_categorical("scale_pos_weight", [1.0, scale_pos_weight]),
    }

    estimator_params = update_estimator_params(
        ml_pipe,
        suggested_estimator_params,
    )
    
    score = ml_pipe.step(
        estimator=ml_pipe._pipeline_params['estimator'],
        estimator_params=estimator_params,
    )
    
    return score

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=xgboost_objective,
    n_trials=n_trials,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-15 12:13:27,517] A new study created in memory with name: model_study
[I 2025-04-15 12:13:27,943] Trial 0 finished with value: 0.6266580841331084 and parameters: {'n_estimators': 406, 'learning_rate': 0.22648248189516848, 'max_depth': 8, 'min_child_weight': 12, 'gamma': 0.7800932022121826, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.5290418060840998, 'reg_alpha': 2.9154431891537547, 'reg_lambda': 0.2537815508265665, 'scale_pos_weight': 1.0}. Best is trial 0 with value: 0.6266580841331084.
[I 2025-04-15 12:13:29,258] Trial 1 finished with value: 0.8286994662609282 and parameters: {'n_estimators': 972, 'learning_rate': 0.11536162338241392, 'max_depth': 4, 'min_child_weight': 4, 'gamma': 0.9170225492671691, 'subsample': 0.6521211214797689, 'colsample_bytree': 0.762378215816119, 'reg_alpha': 0.05342937261279776, 'reg_lambda': 0.014618962793704957, 'scale_pos_weight': 1.0}. Best is trial 1 with value: 0.8286994662609282.
[I 2025-04-15 12:13:29,753] Trial 2 finished wit

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "XGBClassifier"


,0
target,no_fragmentation
model,XGBClassifier_no_fragmentation_opt-model
holdout_test_f1_macro,0.891551
holdout_test_accuracy_balanced,0.936364
holdout_test_roc_auc,0.984545
holdout_test_f1,0.851064
holdout_test_accuracy,0.906667
cv_test_f1_macro_median,0.90293
cv_test_accuracy_balanced_median,0.910256
cv_test_roc_auc_median,0.954212


# AdaBoost

In [ ]:
estimator = AdaBoostClassifier
target = 'splashing'
estimator_params = {}
# params_to_process = ['gamma']
params_to_process = None

def adaboost_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):

    # scale_pos_weight = 132/240 # For splashing

    suggested_estimator_params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True), # Catboost l2_leaf_reg
        # "scale_pos_weight": trial.suggest_categorical("scale_pos_weight", [scale_pos_weight, 1.0]),
    }

    estimator_params = update_estimator_params(
        ml_pipe,
        suggested_estimator_params,
    )
    
    score = ml_pipe.step(
        estimator=ml_pipe._pipeline_params['estimator'],
        estimator_params=estimator_params,
    )
    
    return score

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=adaboost_objective,
    n_trials=n_trials,
)

In [17]:
estimator = AdaBoostClassifier
target = 'splashing'
estimator_params = {}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:34,  1.56it/s]

Progress: 1/54.	Score: 0.8464285714285715.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "AdaBoostClassifier"


,0
target,splashing
model,AdaBoostClassifier_splashing_smotenc_opt-smote...
holdout_test_f1_macro,0.764521
holdout_test_accuracy_balanced,0.760417
holdout_test_roc_auc,0.877701
holdout_test_f1,0.836735
holdout_test_accuracy,0.786667
cv_test_f1_macro_median,0.813161
cv_test_accuracy_balanced_median,0.809598
cv_test_roc_auc_median,0.905573


Model saved in ..\results\models_modelling4\AdaBoostClassifier_splashing_smotenc_opt-smotenc_default-model


In [18]:
estimator = AdaBoostClassifier
target = 'no_fragmentation'
estimator_params = {}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:33,  1.56it/s]

Progress: 1/54.	Score: 0.8250000000000001.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "AdaBoostClassifier"


,0
target,no_fragmentation
model,AdaBoostClassifier_no_fragmentation_smotenc_op...
holdout_test_f1_macro,0.825397
holdout_test_accuracy_balanced,0.852273
holdout_test_roc_auc,0.938182
holdout_test_f1,0.755556
holdout_test_accuracy,0.853333
cv_test_f1_macro_median,0.820946
cv_test_accuracy_balanced_median,0.838828
cv_test_roc_auc_median,0.941392


Model saved in ..\results\models_modelling4\AdaBoostClassifier_no_fragmentation_smotenc_opt-smotenc_default-model


# Random Forest

In [19]:
estimator = RandomForestClassifier
target = 'splashing'
estimator_params = {}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   2%|▏         | 1/54 [00:01<00:53,  1.01s/it]

Progress: 1/54.	Score: 0.8699334543254689.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "RandomForestClassifier"


,0
target,splashing
model,RandomForestClassifier_splashing_smotenc_opt-s...
holdout_test_f1_macro,0.806892
holdout_test_accuracy_balanced,0.799769
holdout_test_roc_auc,0.934799
holdout_test_f1,0.868687
holdout_test_accuracy,0.826667
cv_test_f1_macro_median,0.876935
cv_test_accuracy_balanced_median,0.876935
cv_test_roc_auc_median,0.948142


Model saved in ..\results\models_modelling4\RandomForestClassifier_splashing_smotenc_opt-smotenc_default-model


In [20]:
estimator = RandomForestClassifier
target = 'no_fragmentation'
estimator_params = {}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

Optimizing parameters:   2%|▏         | 1/54 [00:01<00:53,  1.01s/it]

Progress: 1/54.	Score: 0.8299595141700404.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "RandomForestClassifier"


,0
target,no_fragmentation
model,RandomForestClassifier_no_fragmentation_smoten...
holdout_test_f1_macro,0.900794
holdout_test_accuracy_balanced,0.913636
holdout_test_roc_auc,0.98
holdout_test_f1,0.857143
holdout_test_accuracy,0.92
cv_test_f1_macro_median,0.875762
cv_test_accuracy_balanced_median,0.867216
cv_test_roc_auc_median,0.948718


Model saved in ..\results\models_modelling4\RandomForestClassifier_no_fragmentation_smotenc_opt-smotenc_default-model


# LightGBM

In [21]:
estimator = LGBMClassifier
target = 'splashing'
estimator_params = {
    'verbose': -1,
}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:22,  2.40it/s]

Progress: 1/54.	Score: 0.8699334543254689.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "LGBMClassifier"


,0
target,splashing
model,LGBMClassifier_splashing_smotenc_opt-smotenc_d...
holdout_test_f1_macro,0.836601
holdout_test_accuracy_balanced,0.828704
holdout_test_roc_auc,0.939815
holdout_test_f1,0.888889
holdout_test_accuracy,0.853333
cv_test_f1_macro_median,0.896201
cv_test_accuracy_balanced_median,0.900155
cv_test_roc_auc_median,0.94127


Model saved in ..\results\models_modelling4\LGBMClassifier_splashing_smotenc_opt-smotenc_default-model


In [22]:
estimator = LGBMClassifier
target = 'no_fragmentation'
estimator_params = {
    'verbose': -1,
}

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

Optimizing parameters:   2%|▏         | 1/54 [00:00<00:16,  3.26it/s]

Progress: 1/54.	Score: 0.81524926686217.	Considered params: {'k_neighbors': 2, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "LGBMClassifier"


,0
target,no_fragmentation
model,LGBMClassifier_no_fragmentation_smotenc_opt-sm...
holdout_test_f1_macro,0.933862
holdout_test_accuracy_balanced,0.947727
holdout_test_roc_auc,0.99
holdout_test_f1,0.904762
holdout_test_accuracy,0.946667
cv_test_f1_macro_median,0.876543
cv_test_accuracy_balanced_median,0.857143
cv_test_roc_auc_median,0.979487


Model saved in ..\results\models_modelling4\LGBMClassifier_no_fragmentation_smotenc_opt-smotenc_default-model


# For the notebook with Model optimization!

In [23]:
# def update_estimator_params(
#     ml_pipe:MLPipeline,
#     suggested_params:dict,
# ) -> dict:
#     """Update the estimator parameters based on the pipeline parameters.

#     Args:
#         ml_pipe: An instance of MLPipeline used for model training and evaluation.
#         suggested_params: A dictionary containing the suggested Estimator hyperparameters.
    
#     Returns:
#         A dictionary containing the estimator parameters.
#     """
#     estimator_params = ml_pipe._pipeline_params['estimator_params']
#     estimator_params.update(suggested_params)
#     return estimator_params

# def logreg_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):
#     """Objective function for logistic regression optimization using Optuna.

#     Args:
#         trial: An Optuna trial object used to suggest hyperparameters.
#         ml_pipe: An instance of MLPipeline used for model training and evaluation.

#     Returns:
#         The score of the model based on the specified evaluation metric.
#     """
    
#     categorical_features = ('wettability', 'inclination')
    
#     random_state = ml_pipe._pipeline_params['random_state']
    
#     # sample params
#     C = trial.suggest_loguniform('C', 1e-4, 1e3)
    
#     # SMOTE params
#     add_smote = trial.suggest_categorical('add_smote', [True, False])
#     if add_smote:
#         is_smotenc = trial.suggest_categorical('is_smotenc', [True, False])
#         smote_params = {
#             'sampling_strategy': trial.suggest_categorical(
#                 'sampling_strategy', [0.6, 0.7, 0.8, 0.9, 1.0]
#             ),
#             'k_neighbors': trial.suggest_int('k_neighbors', 3, 10),
#             'random_state': random_state,
#         }
#     else:
#         is_smotenc = False
#         smote_params = None
#     if is_smotenc:
#         wettability_cat = trial.suggest_categorical('wettability_cat', [True, False])
#         if wettability_cat:
#             inclination_cat = trial.suggest_categorical('inclination_cat', [True, False])
#         else:
#             inclination_cat = True
        
        
#         mask = [wettability_cat, inclination_cat]
        
#         masked_features = [
#             feature for feature, mask_value in zip(categorical_features, mask) 
#             if mask_value
#         ]
#         smote_params = {
#             **smote_params,
#             'categorical_features': masked_features,
#         }

#     suggested_params = {
#         "C": C,
#     }
    
#     estimator_params = update_estimator_params(ml_pipe, suggested_params)

#     estimator = LogisticRegression(
#         **estimator_params,
#         random_state=random_state,
#     )

#     score = ml_pipe.step(
#         estimator=estimator,
#         add_smote=add_smote,
#         is_smotenc=is_smotenc,
#         smote_params=smote_params,
#     )
    
#     return score



# opt = OptunaOptimizer(
#     objective=partial(logreg_objective, ml_pipe=ml_pipe),
#     study_name="logreg_study",
#     direction="maximize",
# )

# opt.optimize(n_trials=200)

# best_params = opt.study.best_trial.params
# display(best_params)
# # estimator_params = update_estimator_params(ml_pipe, best_params)

# # estimator = LogisticRegression(
# #     **estimator_params,
# #     random_state=ml_pipe._pipeline_params['random_state'],
# # )